# inf-masking — worked example 3: Inf-fill so a min-reduce skips non-hits

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inf-masking`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Beyond attention, the trick overwrites non-candidate entries with `+inf` so a subsequent `min` reduction naturally ignores them — no separate masked-reduce needed. This is the ray-tracing pattern: set the distance of every ray-triangle miss to `+inf`, then take the per-ray minimum to find the nearest hit.

## Worked solution

We find each ray's nearest hit distance, skipping misses.

1. `dists` is `(R, T)` — candidate distances of every ray to every triangle. `hit` is a `(R, T)` bool, `True` where the ray actually hits the triangle.
2. Overwrite misses with `+inf`: `dists.masked_fill(~hit, float('inf'))`. A miss can never be the minimum because `+inf` dominates any real distance.
3. Take the per-ray minimum with `.min(dim=1).values`, shape `(R,)`. Rays with at least one hit return their closest real distance; rays with no hits return `+inf`, a clean sentinel.
4. We verify a known ray's nearest hit and that an all-miss ray yields `+inf`.

In [ ]:
import torch as t

t.manual_seed(2)
dists = t.tensor([[3.0, 1.0, 5.0], [2.0, 4.0, 6.0], [9.0, 9.0, 9.0]])
hit = t.tensor([[True, True, False], [False, True, True], [False, False, False]])

def nearest_hit(dists, hit):
    masked = dists.masked_fill(~hit, float('inf'))
    return masked.min(dim=1).values

near = nearest_hit(dists, hit)
print(near.tolist())
print('ray0 nearest is 1.0:', float(near[0]) == 1.0)
print('ray2 all-miss is inf:', bool(t.isinf(near[2])))